In [ ]:
# 1. Install required dependencies
!pip install ultralytics -q

import os
import glob
from ultralytics import YOLO
from IPython.display import Image, display

# 2. Unzip the Dataset
# Change 'pascal_voc.zip' if your file has a different name
zip_path = '/content/pascal_voc.zip'
dataset_dir = '/content/dataset'

print("Unzipping dataset...")
!unzip -q {zip_path} -d {dataset_dir}
print("Unzip complete.")

# Roboflow datasets typically contain a data.yaml file at the root of the unzipped folder.
# Let's verify its path.
yaml_path = f'{dataset_dir}/data.yaml'
if not os.path.exists(yaml_path):
    print(f"Warning: data.yaml not found at {yaml_path}. Please check your unzipped folder structure.")

# 3. Initialize and Train the Base Model
print("Initializing YOLOv8n baseline...")
model = YOLO('yolov8n.pt') # Loads the pretrained nano model weights

print("Starting training phase...")
# Training parameters. Adjust epochs based on your time constraints.
results = model.train(
    data=yaml_path,
    epochs=50,       # Start with 50 for a solid baseline
    imgsz=640,       # Standard YOLO resolution
    batch=16,        # Good default for Colab GPUs
    project='adaptive_system',
    name='baseline_yolov8n',
    exist_ok=True
)

# 4. Validation & Metrics
print("\n--- Model Evaluation ---")
# Ultralytics automatically runs validation at the end of training,
# but calling val() explicitly lets us grab the metrics object easily.
metrics = model.val()

print(f"mAP50-95 (Overall Accuracy): {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}") # Mean precision across classes
print(f"Recall: {metrics.box.mr:.4f}")    # Mean recall across classes

# Note on F1 Score:
# Ultralytics calculates the F1 score across all confidence thresholds and generates an F1_curve.png
# You can view the optimal F1 score graph saved in the run directory.
f1_curve_path = '/content/adaptive_system/baseline_yolov8n/F1_curve.png'
if os.path.exists(f1_curve_path):
    print("\nDisplaying F1-Confidence Curve:")
    display(Image(filename=f1_curve_path))

# 5. Inference / Testing
print("\n--- Running Inference ---")
# Find a sample image from the test set
test_images = glob.glob(f'{dataset_dir}/test/images/*.jpg')

if test_images:
    sample_img = test_images[0]
    print(f"Running inference on {sample_img}...")

    # Run prediction
    infer_results = model.predict(
        source=sample_img,
        conf=0.25, # Confidence threshold
        save=True,
        project='adaptive_system',
        name='inference_test'
    )

    # Display the result
    result_img_path = glob.glob('/content/adaptive_system/inference_test/*.jpg')[0]
    display(Image(filename=result_img_path))
else:
    print("No test images found. Check if your dataset has a 'test/images' split.")